In [1]:
import os
import json

# 1. Start in the base Colab directory
%cd /content

# 2. Clone the repository
if not os.path.exists('/content/VisAttnSink'):
    !git clone https://github.com/seilk/VisAttnSink.git

# 3. Enter the project folder
%cd /content/VisAttnSink

# 4. FIX DEPENDENCIES (Strip out ALL problematic packages, including ml-dtypes)
print("Fixing dependencies...")
!sed -i 's/.*@ file:\/\/\/.*//g' env_pip.txt
!sed -i '/torch==/d' env_pip.txt
!sed -i '/torchvision==/d' env_pip.txt
!sed -i '/torchaudio==/d' env_pip.txt
!sed -i '/mkl-service==/d' env_pip.txt
!sed -i '/flash-attn==/d' env_pip.txt
!sed -i '/ml-dtypes/d' env_pip.txt  # Removes ml-dtypes from the list!

# 5. Install the repository packages
print("Installing packages...")
!pip install -r env_pip.txt --no-build-isolation

# 6. FORCE UPGRADE ml_dtypes AFTER everything else
print("Forcing JAX/ml_dtypes compatibility...")
!pip install ml-dtypes>=0.5.0 --upgrade

# 7. Create dataset directory structure
print("Setting up folders and test data...")
!mkdir -p C_datasets/TestDataset/Images
!mkdir -p C_datasets/TestDataset/Questions
!mkdir -p A_exps

# 8. Download a test image
!wget -q https://raw.githubusercontent.com/haotian-liu/LLaVA/main/images/llava_logo.png -O C_datasets/TestDataset/Images/llava_logo.png

# 9. Create the JSONL questions file
questions = [
    {"question_id": 1, "image": "llava_logo.png", "text": "What is depicted in this image?", "label": "A logo for LLaVA"},
    {"question_id": 2, "image": "llava_logo.png", "text": "Is this a photograph or a graphic? Explain your reasoning.", "label": "A graphic"}
]
with open("C_datasets/TestDataset/Questions/General-questions.jsonl", "w") as f:
    for q in questions:
        f.write(json.dumps(q) + "\n")

# 10. Create the YAML configuration file
yaml_config = """
name_exp: test_colab_run
name_daset: TestDataset
name_category: General

path_image_dir: C_datasets/TestDataset/Images
path_question_dir: C_datasets/TestDataset/Questions
path_model: liuhaotian/llava-v1.5-7b

conv_mode: vicuna_v1

logic: 1
tau: 20
rho: 0.5
p: 0.6
summ: 0.2

max_new_tokens: 128
except_last_layer: 1
"""
with open("A_exps/colab_test.yml", "w") as f:
    f.write(yaml_config.strip())

# 11. Run the inference!
print("Starting inference...")
!python src/inference.py --device 0 --exp_config A_exps/colab_test.yml

/content
/content/VisAttnSink
Fixing dependencies...
Installing packages...
  Using cached segment_anything-1.0-py3-none-any.whl
  Using cached transformers-4.47.0.dev0-py3-none-any.whl
Forcing JAX/ml_dtypes compatibility...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires ml-dtypes<0.5.0,>=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.17.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.17.0 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.17.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [2]:
# Unzip MME subset
!unzip -q /content/MME_Benchmark_Subset.zip -d /content/MME_Benchmark_Subset

# Unzip POPE subset
!unzip -q /content/POPE_Final_Subset.zip -d /content/POPE_Final_Subset

# Unzip MMVP subset
!unzip -q /content/mmvp_subset_50.zip -d /content/mmvp_subset_50

In [5]:
# configure for MMVP

yaml_config = """
name_exp: MMVP_Run
name_daset: MMVP
name_category: mmvp

path_image_dir: C_datasets/MMVP/Images
path_question_dir: C_datasets/MMVP/Questions
path_model: liuhaotian/llava-v1.5-7b

conv_mode: vicuna_v1

logic: 1
tau: 20
rho: 0.9
p: 0.6
summ: 0.2

max_new_tokens: 128
except_last_layer: 1
"""

with open("A_exps/lv1.5_7b.yml", "w") as f:
    f.write(yaml_config.strip())


print("Starting inference...")
!python src/inference.py --device 0 --exp_config A_exps/lv1.5_7b.yml



Starting inference...
2026-03-24 19:43:46.070205: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 19:43:46.087546: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-24 19:43:46.108017: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-24 19:43:46.114249: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-24 19:43:46.128853: I tensorflow/core/pla

In [10]:
# configure for MME

yaml_config = """
name_exp: MME_Run
name_daset: MME
name_category: mme

path_image_dir: C_datasets/MME/Images
path_question_dir: C_datasets/MME/Questions
path_model: liuhaotian/llava-v1.5-7b

conv_mode: vicuna_v1

logic: 1
tau: 20
rho: 0.8
p: 0.6
summ: 0.2

max_new_tokens: 128
except_last_layer: 1
"""

with open("A_exps/MME.yml", "w") as f:
    f.write(yaml_config.strip())


print("Starting inference...")
!python src/inference.py --device 0 --exp_config A_exps/MME.yml



Starting inference...
2026-03-24 20:41:25.259634: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 20:41:25.277118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-24 20:41:25.297939: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-24 20:41:25.304278: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-24 20:41:25.319025: I tensorflow/core/pla

In [11]:
# configure for POPE

yaml_config = """
name_exp: POPE_Run
name_daset: POPE
name_category: pope

path_image_dir: C_datasets/POPE/Images
path_question_dir: C_datasets/POPE/Questions
path_model: liuhaotian/llava-v1.5-7b

conv_mode: vicuna_v1

logic: 1
tau: 20
rho: 0.5
p: 0.6
summ: 0.2

max_new_tokens: 128
except_last_layer: 1
"""

with open("A_exps/POPE.yml", "w") as f:
    f.write(yaml_config.strip())


print("Starting inference...")
!python src/inference.py --device 0 --exp_config A_exps/POPE.yml

Starting inference...
2026-03-24 20:47:10.226385: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 20:47:10.245434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-24 20:47:10.267026: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-24 20:47:10.273671: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-24 20:47:10.290511: I tensorflow/core/pla

In [19]:
import os
import json

# Let's test all the likely exact paths based on your Colab environment
possible_paths = [
    "/content/VisAttnSink/E_answers/llava-v1.5-7b/[POPE-pope]POPE_Run-1774385245.jsonl",
    "VisAttnSink/E_answers/llava-v1.5-7b/[POPE-pope]POPE_Run-1774385245.jsonl",
    "E_answers/llava-v1.5-7b/[POPE-pope]POPE_Run-1774385245.jsonl",
    # Just in case the capitalization is slightly different:
    "E_answers/llava-v1.5-7b/[Pope-pope]POPE_Run-1774385245.jsonl"
]

result_file = None
for path in possible_paths:
    if os.path.exists(path):
        result_file = path
        break

if not result_file:
    print("Could not find the POPE jsonl file.")
    print("Please run this command in a new cell to see the exact filename:")
    print("!ls -la /content/VisAttnSink/E_answers/llava-v1.5-7b/")
else:
    print(f"✅ Found file! Evaluating: {result_file}\n")

    answers = [json.loads(line) for line in open(result_file)]

    pred_list = []
    label_list = []

    for item in answers:
        # Extract Ground Truth
        gt = str(item.get('label', '')).lower().strip()
        label_list.append(0 if gt == 'no' else 1)

        # Extract Model Prediction
        text = str(item.get('response', item.get('text', ''))).strip()

        # Only keep the first sentence for POPE formatting
        if text.find('.') != -1:
            text = text.split('.')[0]

        text = text.replace(',', '')
        words = text.split(' ')

        # Determine Yes/No
        if 'No' in words or 'not' in words or 'no' in words:
            pred_list.append(0)
        else:
            pred_list.append(1)

    pos, neg = 1, 0
    yes_ratio = pred_list.count(1) / len(pred_list) if pred_list else 0

    TP, TN, FP, FN = 0, 0, 0, 0
    for pred, label in zip(pred_list, label_list):
        if pred == pos and label == pos:
            TP += 1
        elif pred == pos and label == neg:
            FP += 1
        elif pred == neg and label == neg:
            TN += 1
        elif pred == neg and label == pos:
            FN += 1

    print('TP\tFP\tTN\tFN')
    print(f'{TP}\t{FP}\t{TN}\t{FN}\n')

    precision = float(TP) / float(TP + FP) if (TP + FP) > 0 else 0
    recall = float(TP) / float(TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    acc = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1 score:  {f1:.4f}')
    print(f'Yes ratio: {yes_ratio:.4f}')

✅ Found file! Evaluating: /content/VisAttnSink/E_answers/llava-v1.5-7b/[POPE-pope]POPE_Run-1774385245.jsonl

TP	FP	TN	FN
38	6	34	2

Accuracy:  0.9000
Precision: 0.8636
Recall:    0.9500
F1 score:  0.9048
Yes ratio: 0.5500


In [20]:
import json
import os
import glob
from collections import defaultdict

# 1. Find the MME jsonl file
mme_files = glob.glob("E_answers/**/*MME*.jsonl", recursive=True)

if not mme_files:
    print("Could not find the MME jsonl file. Please check the path!")
else:
    # Use the file with the most lines (in case there are leftover incomplete runs)
    mme_files.sort(key=lambda x: os.path.getsize(x), reverse=True)
    result_file = mme_files[0]
    print(f"Evaluating MME Benchmark for: {result_file}\n")
    print("-" * 50)

    # 2. Parse the answers
    answers = [json.loads(line) for line in open(result_file)]

    # We will group by the image file name, because MME has 2 questions per image
    # structure: category -> image_file -> list of boolean results
    results_by_category = defaultdict(lambda: defaultdict(list))

    for item in answers:
        # Extract ground truth and convert to lowercase
        gt = str(item.get('label', item.get('answer', ''))).lower().strip()

        # Extract model response
        text = str(item.get('response', item.get('text', ''))).lower().strip()

        # Standard MME cleaning: Keep the first sentence/word to isolate "yes" or "no"
        if text.find('.') != -1:
            text = text.split('.')[0]
        text = text.replace(',', '').split(' ')

        if 'no' in text or 'not' in text:
            pred = 'no'
        else:
            pred = 'yes'

        is_correct = (pred == gt)

        # Extract Category and Image ID
        # If image path is 'existence/02598153523880aa.jpg', category is 'existence'
        # If there's no folder path, we'll group them all under 'Overall'
        image_path = item.get('image', 'Overall/unknown.jpg')

        if '/' in image_path:
            category = image_path.split('/')[-2] # Get the folder name
            img_id = image_path.split('/')[-1]
        else:
            category = 'Overall'
            img_id = image_path

        # Also fallback: sometimes the category is explicitly in the jsonl
        if 'category' in item:
            category = item['category']

        results_by_category[category][img_id].append(is_correct)

    # 3. Calculate and print scores per category
    total_acc_sum = 0
    total_acc_plus_sum = 0

    print(f"{'Category':<20} | {'Accuracy':<10} | {'Accuracy+':<10} | {'Score (Max 200)':<10}")
    print("-" * 60)

    for category, images in results_by_category.items():
        total_questions = 0
        correct_questions = 0

        total_images = 0
        correct_images = 0

        for img_id, outcomes in images.items():
            total_images += 1
            total_questions += len(outcomes)
            correct_questions += sum(outcomes)

            # Accuracy+ requires BOTH questions for the image to be correct
            if len(outcomes) == 2 and sum(outcomes) == 2:
                correct_images += 1

        # Calculate percentages
        acc = (correct_questions / total_questions) * 100 if total_questions > 0 else 0
        acc_plus = (correct_images / total_images) * 100 if total_images > 0 else 0
        score = acc + acc_plus

        total_acc_sum += acc
        total_acc_plus_sum += acc_plus

        print(f"{category:<20} | {acc:<10.2f} | {acc_plus:<10.2f} | {score:<10.2f}")

    print("-" * 60)

    # 4. Total MME Score across all present categories
    num_categories = len(results_by_category)
    if num_categories > 0:
        avg_acc = total_acc_sum / num_categories
        avg_acc_plus = total_acc_plus_sum / num_categories
        total_score = total_acc_sum + total_acc_plus_sum
        print(f"{'TOTAL / AVERAGE':<20} | {avg_acc:<10.2f} | {avg_acc_plus:<10.2f} | {total_score:<10.2f}")

Evaluating MME Benchmark for: E_answers/llava-v1.5-7b/[MME-mme]MME_Run-1774384900.jsonl

--------------------------------------------------
Category             | Accuracy   | Accuracy+  | Score (Max 200)
------------------------------------------------------------
Overall              | 51.00      | 4.00       | 55.00     
------------------------------------------------------------
TOTAL / AVERAGE      | 51.00      | 4.00       | 55.00     


### MMVP Evaluation with Gemini 3.1 Pro

**1. Evaluation Summary**
- **Total Questions:** 300
- **Correct Answers:** 181
- **Incorrect Answers:** 119
- **Final Accuracy: 60.33%**

**2. Error Analysis**

| Question ID | Model's Raw Prediction | Normalized Choice | Ground Truth | Options |
|:---|:---|:---:|:---:|:---|
| 3 | The flame of the match is thin. | (b) | (a) | (a) round (b) thin |
| 5 | (b) Right | (b) | (a) | (a) left (b) right |
| 8 | (a) Towards the camera | (a) | (b) | (a) towards the camera (b) away from the camera |
| 12 | Yes | (a) | (b) | (a) yes (b) no |
| 13 | The image is more likely to show multiple ears of corn... | (b) | (a) | (a) single (b) multiple |
| 16 | Yes | (a) | (b) | (a) yes (b) no |
| 18 | Yes | (a) | (b) | (a) yes (b) no |
| 20 | Yes | (a) | (b) | (a) yes (b) no |
| 21 | No | (b) | (a) | (a) yes (b) no |
| 23 | (b) Side | (b) | (a) | (a) front (b) side |
| 29 | (b) Incorrect | (b) | (a) | (a) correct (b) incorrect |
| 32 | Yes | (a) | (b) | (a) yes (b) no |
| 34 | Yes | (a) | (b) | (a) yes (b) no |
| 35 | No | (b) | (a) | (a) yes (b) no |
| 41 | (b) No | (b) | (a) | (a) yes (b) no |
| 42 | Yes | (a) | (b) | (a) yes (b) no |
| 47 | The animal in the image has 6 spots. | (b) | (a) | (a) 5 (b) 6 |

*(Table truncated to top instances for readability)*

**Observations for further investigation:**
1. **Directional/Spatial Confusion:** The model frequently struggles with relative spatial orientation (e.g., Up vs. Down, Inward vs Outward, Towards vs. Away). 
2. **Yes/No Hallucinations:** The model has a significant bias towards answering "Yes" or erroneously tracking visual presence when items are occluded or missing.
3. **Format Slippage:** Certain predictions semantically identify the correct option but missed out on being counted because the model failed to explicitly output the digit or the exact letter as instructed in the testing protocol.